# Reviewer 2 matched, leakage-safe evaluation

This notebook is a clean evaluation and retraining companion to `ML2_COMPLETION (1).ipynb`. It does not overwrite or optimize for historical scores.

Protocol:
- use the complete paired QTDB population and split records before creating windows (`test_size=0.20`, seed 7);
- split LUDB records before creating windows (20 adaptation, 180 untouched test, seed 17);
- compute normalization only from QTDB training windows;
- evaluate valid existing checkpoints on the common LUDB test records;
- retrain R1-R6 for five independent neural-network seeds;
- report fixed-model record bootstrap separately from training-seed mean, SD, and t-based 95% CI.

Run cells top to bottom. The five-seed block is the authoritative repeated-run experiment. The legacy adapted R6 checkpoint is used only if an accompanying provenance manifest proves that it used the exact current 20-record adaptation list.

In [4]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, find_peaks, resample_poly
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
import wfdb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'PQRST_mapping').exists() and (REPO_ROOT / 'data').exists():
    REPO_ROOT = REPO_ROOT.parent

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])

# Prefer the complete PhysioNet copy; data/qtdb is only a seven-record subset.
QTDB_DIR = first_existing(
    REPO_ROOT / 'physionet.org/files/qtdb/1.0.0',
    REPO_ROOT / 'PQRST_mapping/physionet.org/files/qtdb/1.0.0',
    REPO_ROOT / 'data/qtdb',
    REPO_ROOT / 'PQRST_mapping/data/qtdb',
    Path('/kaggle/input/datasets/vrishankamembal/qtdb-1-0-0/qt-database-1.0.0'),
)
LUDB_DIR = first_existing(
    REPO_ROOT / 'data/ludb',
    REPO_ROOT / 'PQRST_mapping/data/ludb',
    REPO_ROOT / 'physionet.org/files/ludb/1.0.1/data',
    REPO_ROOT / 'PQRST_mapping/physionet.org/files/ludb/1.0.1/data',
    Path('/kaggle/input/datasets/vrishankamembal/ludb-1-0-1/lobachevsky-university-electrocardiography-database-1.0.1/data'),
)
ARTIFACT_DIR = REPO_ROOT / 'PQRST_mapping/reviewer2_artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
PRE, POST, R5_POST = 120, 240, 320
BATCH_SIZE = 64
NUM_EPOCHS = 30
TRAINING_SEEDS = [1, 2, 3, 4, 5]
R6_ADAPT_FRACTION, R6_SPLIT_SEED, R6_ADAPT_EPOCHS = 0.10, 17, 8
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINTS = {
    'R1': 'best_ml2_rpeak_guided.pth',
    'R2': 'best_ml2_rpeak_guided_focal.pth',
    'R3': 'best_ml2_rpeak_guided_focal_unweighted.pth',
    'R5': 'best_ml2_r5_post320.pth',
    'R6 QTDB': 'best_ml2_r6_time_qtdb.pth',
    'R6 adapted': 'best_ml2_r6_time_ludb10.pth',
}
print('QTDB:', QTDB_DIR)
print('LUDB:', LUDB_DIR)
print('Device:', DEVICE)

QTDB: /Users/vishwam/VSCode/physionet.org/files/qtdb/1.0.0
LUDB: /Users/vishwam/VSCode/PQRST_mapping/physionet.org/files/ludb/1.0.1/data
Device: mps


## Shared labels, detector, and window builder

In [5]:
def generate_labels(annotation, length, sample_scale=1.0):
    labels = np.zeros(length, dtype=np.int64)
    start, wave_kind = None, None
    for sample, symbol in zip(annotation.sample, annotation.symbol):
        sample = int(round(sample * sample_scale))
        if symbol == '(': start, wave_kind = sample, None
        elif symbol == 'p': wave_kind = 1
        elif symbol == 'N': wave_kind = 2
        elif symbol == 't': wave_kind = 3
        elif symbol == ')' and start is not None and wave_kind is not None:
            labels[max(0, start):min(length, sample + 1)] = wave_kind
            start, wave_kind = None, None
    labels[labels == 2] = 0
    labels[labels == 3] = 2
    return labels

def pan_tompkins_r_peaks(ecg, fs):
    nyquist = fs / 2
    sos = butter(3, [5.0 / nyquist, 18.0 / nyquist], btype='bandpass', output='sos')
    bandpassed = sosfiltfilt(sos, ecg)
    derivative = np.convolve(bandpassed, np.array([-1, -2, 0, 2, 1]) * fs / 8, mode='same')
    width = max(1, round(0.150 * fs))
    integrated = np.convolve(derivative ** 2, np.ones(width) / width, mode='same')
    candidates, _ = find_peaks(integrated, distance=max(1, round(0.20 * fs)))
    boot = candidates[candidates < min(len(ecg), round(2 * fs))]
    spki = np.percentile(integrated[boot], 90) if len(boot) else 0.0
    npki = np.percentile(integrated[boot], 25) if len(boot) else 0.0
    accepted = []
    for peak in candidates:
        threshold = npki + 0.25 * (spki - npki)
        if integrated[peak] >= threshold:
            accepted.append(peak); spki = 0.125 * integrated[peak] + 0.875 * spki
        else:
            npki = 0.125 * integrated[peak] + 0.875 * npki
    search = round(0.10 * fs)
    refined = []
    for peak in accepted:
        left, right = max(0, peak - search), min(len(ecg), peak + search + 1)
        refined.append(left + np.argmax(ecg[left:right]))
    return np.unique(np.asarray(refined, dtype=int))

def create_windows(ecg, labels, r_peaks, post):
    x, y = [], []
    for r in r_peaks:
        left, right = r - PRE, r + post
        if left >= 0 and right <= len(ecg):
            x.append(ecg[left:right]); y.append(labels[left:right])
    return np.asarray(x, dtype=np.float32), np.asarray(y, dtype=np.int64)

def qtdb_record(record_name, post):
    path = str(QTDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    ecg = record.p_signal[:, 0].astype(np.float32)
    labels = generate_labels(wfdb.rdann(path, 'pu0'), len(ecg))
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, float(record.fs)), post)

def ludb_record(record_name, post):
    path = str(LUDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    lead = record.sig_name.index('ii')
    ecg = resample_poly(record.p_signal[:, lead].astype(np.float32), up=1, down=2)
    labels = generate_labels(wfdb.rdann(path, 'ii'), len(ecg), sample_scale=0.5)
    size = min(len(ecg), len(labels))
    ecg, labels = ecg[:size], labels[:size]
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, 250.0), post)

def build_partition(record_names, builder, post):
    xs, ys, ids, skipped = [], [], [], []
    for record_name in record_names:
        try:
            x, y = builder(record_name, post)
            if len(x):
                xs.append(x); ys.append(y); ids.extend([record_name] * len(x))
        except Exception as error:
            skipped.append({'record': record_name, 'error': str(error)})
    if not xs:
        raise RuntimeError('No usable windows were generated')
    return np.concatenate(xs), np.concatenate(ys), np.asarray(ids), skipped

## Record-level partitions and train-only normalization

In [6]:
qtdb_headers = {path.stem for path in QTDB_DIR.glob('*.hea')}
qtdb_dat = {path.stem for path in QTDB_DIR.glob('*.dat')}
qtdb_records = sorted(qtdb_headers & qtdb_dat)
qtdb_unpaired = sorted(qtdb_headers ^ qtdb_dat)
if qtdb_unpaired:
    print('Unpaired QTDB files:', qtdb_unpaired)
assert len(qtdb_records) == 105, f'Expected 105 paired QTDB records, found {len(qtdb_records)} in {QTDB_DIR}'
qtdb_train_records, qtdb_val_records = train_test_split(qtdb_records, test_size=0.20, random_state=7, shuffle=True)
qtdb_train_records, qtdb_val_records = sorted(qtdb_train_records), sorted(qtdb_val_records)
assert len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21
assert set(qtdb_train_records).isdisjoint(qtdb_val_records)
assert set(qtdb_train_records) | set(qtdb_val_records) == set(qtdb_records)

# Windows are created independently after the record split.
X_qt_train, Y_qt_train, qt_train_ids, qt_train_skipped = build_partition(qtdb_train_records, qtdb_record, POST)
X_qt_val, Y_qt_val, qt_val_ids, qt_val_skipped = build_partition(qtdb_val_records, qtdb_record, POST)
train_mean = float(X_qt_train.mean())
train_std = float(X_qt_train.std())
if train_std == 0: raise ValueError('QTDB training standard deviation is zero')
X_qt_train = (X_qt_train - train_mean) / train_std
X_qt_val = (X_qt_val - train_mean) / train_std

ludb_headers = {path.stem for path in LUDB_DIR.glob('*.hea')}
ludb_dat = {path.stem for path in LUDB_DIR.glob('*.dat')}
ludb_records = sorted(ludb_headers & ludb_dat)
ludb_unpaired = sorted(ludb_headers ^ ludb_dat)
assert len(ludb_records) == 200, f'Expected 200 paired LUDB records, found {len(ludb_records)} in {LUDB_DIR}'
r6_adapt_records, r6_test_records = train_test_split(ludb_records, test_size=1 - R6_ADAPT_FRACTION, random_state=R6_SPLIT_SEED, shuffle=True)
r6_adapt_records, r6_test_records = sorted(r6_adapt_records), sorted(r6_test_records)
assert len(r6_adapt_records) == 20 and len(r6_test_records) == 180
assert set(r6_adapt_records).isdisjoint(r6_test_records)
assert set(r6_adapt_records) | set(r6_test_records) == set(ludb_records)

# Build adaptation and untouched test windows separately, for each model window length.
X_lu_adapt_240, Y_lu_adapt_240, lu_adapt_ids_240, lu_adapt_skipped_240 = build_partition(r6_adapt_records, ludb_record, POST)
X_lu_test_240, Y_lu_test_240, lu_test_ids_240, lu_test_skipped_240 = build_partition(r6_test_records, ludb_record, POST)
X_lu_adapt_320, Y_lu_adapt_320, lu_adapt_ids_320, lu_adapt_skipped_320 = build_partition(r6_adapt_records, ludb_record, R5_POST)
X_lu_test_320, Y_lu_test_320, lu_test_ids_320, lu_test_skipped_320 = build_partition(r6_test_records, ludb_record, R5_POST)

assert set(qt_train_ids).isdisjoint(qt_val_ids)
assert set(lu_adapt_ids_240).isdisjoint(lu_test_ids_240)
assert set(lu_adapt_ids_320).isdisjoint(lu_test_ids_320)
assert set(np.unique(qt_train_ids)) == set(qtdb_train_records) - {row['record'] for row in qt_train_skipped}
assert set(np.unique(qt_val_ids)) == set(qtdb_val_records) - {row['record'] for row in qt_val_skipped}

split_manifest = {
    'qtdb_directory': str(QTDB_DIR),
    'ludb_directory': str(LUDB_DIR),
    'qtdb_total_records': len(qtdb_records),
    'qtdb_train_records': qtdb_train_records, 'qtdb_validation_records': qtdb_val_records,
    'qtdb_train_windows': len(X_qt_train), 'qtdb_validation_windows': len(X_qt_val),
    'qtdb_skipped': qt_train_skipped + qt_val_skipped,
    'ludb_total_records': len(ludb_records),
    'ludb_adaptation_records': r6_adapt_records, 'ludb_untouched_test_records': r6_test_records,
    'ludb_adaptation_windows_240': len(X_lu_adapt_240), 'ludb_test_windows_240': len(X_lu_test_240),
    'ludb_adaptation_windows_320': len(X_lu_adapt_320), 'ludb_test_windows_320': len(X_lu_test_320),
    'ludb_skipped': lu_adapt_skipped_240 + lu_test_skipped_240,
    'qtdb_train_mean': train_mean, 'qtdb_train_std': train_std,
    'qtdb_split_seed': 7, 'ludb_split_seed': R6_SPLIT_SEED,
}
(ARTIFACT_DIR / 'reviewer2_split_manifest.json').write_text(json.dumps(split_manifest, indent=2))
np.savez(ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz', mean=train_mean, std=train_std)
print(f'QTDB total records: {len(qtdb_records)} | train: {len(qtdb_train_records)} | validation: {len(qtdb_val_records)}')
print('QTDB train records:', qtdb_train_records)
print('QTDB validation records:', qtdb_val_records)
print(f'LUDB total records: {len(ludb_records)} | adaptation: {len(r6_adapt_records)} | test: {len(r6_test_records)}')
print('LUDB adaptation records:', r6_adapt_records)
print('LUDB test records:', r6_test_records)
print('QTDB skipped:', qt_train_skipped + qt_val_skipped)
print('LUDB skipped:', lu_adapt_skipped_240 + lu_test_skipped_240)
print('Windows:', len(X_qt_train), len(X_qt_val), len(X_lu_adapt_240), len(X_lu_test_240), len(X_lu_adapt_320), len(X_lu_test_320))
print('Normalization: mean=', train_mean, 'std=', train_std)

QTDB total records: 105 | train: 84 | validation: 21
QTDB train records: ['sel100', 'sel102', 'sel103', 'sel104', 'sel114', 'sel117', 'sel123', 'sel14046', 'sel14157', 'sel14172', 'sel15814', 'sel16273', 'sel16420', 'sel16483', 'sel16539', 'sel16773', 'sel16786', 'sel17152', 'sel213', 'sel221', 'sel223', 'sel231', 'sel232', 'sel233', 'sel30', 'sel301', 'sel302', 'sel306', 'sel307', 'sel308', 'sel31', 'sel310', 'sel32', 'sel33', 'sel34', 'sel35', 'sel36', 'sel37', 'sel38', 'sel39', 'sel41', 'sel42', 'sel45', 'sel46', 'sel47', 'sel48', 'sel49', 'sel50', 'sel51', 'sel52', 'sel808', 'sel811', 'sel820', 'sel821', 'sel840', 'sel847', 'sel853', 'sel871', 'sel872', 'sel873', 'sel883', 'sele0104', 'sele0106', 'sele0110', 'sele0114', 'sele0116', 'sele0121', 'sele0124', 'sele0126', 'sele0129', 'sele0133', 'sele0136', 'sele0166', 'sele0170', 'sele0203', 'sele0210', 'sele0303', 'sele0405', 'sele0409', 'sele0603', 'sele0604', 'sele0606', 'sele0607', 'sele0612']
QTDB validation records: ['sel116', 's

## Models, locked R4 decoder, and checkpoint scoring

In [7]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)

class BiLSTMBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)

    def forward(self, x):
        return self.lstm(x)[0]

class RPeakGuidedML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        self.bilstm = BiLSTMBlock()
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)))

class RPeakTimeML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.bilstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)[0]))

def time_channel(signals, post):
    time = (np.arange(signals.shape[1], dtype=np.float32) - PRE) / post
    return np.stack([signals.astype(np.float32), np.broadcast_to(time, signals.shape)], axis=1).copy()

def predict(model, features):
    model.eval(); predictions = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            predictions.append(model(batch).argmax(2).cpu().numpy())
    return np.concatenate(predictions)

def predict_probabilities(model, features):
    model.eval(); probabilities = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            probabilities.append(torch.softmax(model(batch), dim=2).cpu().numpy())
    return np.concatenate(probabilities)

R4_T_START, R4_T_MIN_LENGTH = PRE + 5, 4
def r4_decode(probabilities, threshold):
    decoded = probabilities.argmax(axis=2).astype(np.int64)
    for beat, probs in enumerate(probabilities):
        decoded[beat, PRE:] = np.where(decoded[beat, PRE:] == 1, 0, decoded[beat, PRE:])
        decoded[beat, decoded[beat] == 2] = 0
        active = np.zeros(len(probs), dtype=bool); active[R4_T_START:] = probs[R4_T_START:, 2] >= threshold
        edges = np.diff(np.r_[False, active, False].astype(int))
        choices = [(a, b - 1) for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)) if b - a >= R4_T_MIN_LENGTH]
        if choices:
            a, b = max(choices, key=lambda part: probs[part[0]:part[1] + 1, 2].sum()); decoded[beat, a:b + 1] = 2
    return decoded

def classification_row(name, truth, prediction, records, post):
    report = classification_report(truth.ravel(), prediction.ravel(), labels=[0, 1, 2], target_names=['Background', 'P Wave', 'T Wave'], output_dict=True, zero_division=0)
    return {'Experiment': name, 'Records': len(np.unique(records)), 'Windows': len(truth), 'Post samples': post, 'Accuracy': accuracy_score(truth.ravel(), prediction.ravel()), 'P precision': report['P Wave']['precision'], 'P recall': report['P Wave']['recall'], 'P F1': report['P Wave']['f1-score'], 'T precision': report['T Wave']['precision'], 'T recall': report['T Wave']['recall'], 'T F1': report['T Wave']['f1-score'], 'Macro F1': report['macro avg']['f1-score'], 'Weighted F1': report['weighted avg']['f1-score']}

def load_checkpoint(path, time=False):
    model = (RPeakTimeML2() if time else RPeakGuidedML2()).to(DEVICE)
    checkpoint = Path(path)
    if not checkpoint.is_absolute(): checkpoint = Path.cwd() / checkpoint
    if not checkpoint.exists(): checkpoint = Path(REPO_ROOT) / 'PQRST_mapping' / path
    if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    return model

# R4 threshold is selected on QTDB validation only. LUDB labels are not read in this cell.
r4_base = load_checkpoint(CHECKPOINTS['R1'])
r4_val_probs = predict_probabilities(r4_base, X_qt_val[:, None])
threshold_rows = []
for threshold in np.arange(0.10, 0.76, 0.05):
    pred = r4_decode(r4_val_probs, threshold)
    threshold_rows.append((threshold, f1_score(Y_qt_val.ravel(), pred.ravel(), labels=[0, 1, 2], average='macro', zero_division=0)))
R4_T_THRESHOLD = float(max(threshold_rows, key=lambda row: row[1])[0])
print('R4 threshold locked from QTDB validation:', R4_T_THRESHOLD)

R4 threshold locked from QTDB validation: 0.5000000000000001


In [9]:
from scipy.stats import t as student_t


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, target):
        log_probability = torch.log_softmax(logits, dim=1)
        probability = log_probability.exp()
        focal = (1.0 - probability) ** self.gamma
        if self.alpha is not None:
            focal = focal * self.alpha.view(1, -1, 1)
        return (-focal * log_probability).gather(1, target.unsqueeze(1)).mean()


def make_loader(features, labels, seed, shuffle):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        TensorDataset(torch.as_tensor(features, dtype=torch.float32), torch.as_tensor(labels, dtype=torch.long)),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )


def run_training_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, batches = 0.0, 0
    with torch.set_grad_enabled(training):
        for features, labels in loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            loss = criterion(model(features).permute(0, 2, 1), labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total += float(loss.item())
            batches += 1
    return total / max(1, batches)


def train_fixed_model(model, train_features, train_labels, val_features, val_labels, criterion, seed, run_name):
    set_global_seed(seed)
    train_loader = make_loader(train_features, train_labels, seed, shuffle=True)
    val_loader = make_loader(val_features, val_labels, seed, shuffle=False)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss, best_epoch, best_state = float('inf'), 0, None
    for epoch in range(NUM_EPOCHS):
        train_loss = run_training_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_training_epoch(model, val_loader, criterion)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        print(f'{run_name} seed={seed} epoch={epoch + 1:02d}: train={train_loss:.4f} val={val_loss:.4f}')
    model.load_state_dict(best_state)
    return model, {'best_epoch': best_epoch, 'best_validation_loss': best_loss}


# R5/R6 use their 440-sample representation and normalization from QTDB training windows only.
X_qt_train_320, Y_qt_train_320, qt_train_ids_320, qt_train_skipped_320 = build_partition(qtdb_train_records, qtdb_record, R5_POST)
X_qt_val_320, Y_qt_val_320, qt_val_ids_320, qt_val_skipped_320 = build_partition(qtdb_val_records, qtdb_record, R5_POST)
r5_train_mean, r5_train_std = float(X_qt_train_320.mean()), float(X_qt_train_320.std())
X_qt_train_320 = (X_qt_train_320 - r5_train_mean) / r5_train_std
X_qt_val_320 = (X_qt_val_320 - r5_train_mean) / r5_train_std
X_lu_adapt_240 = (X_lu_adapt_240 - train_mean) / train_std
X_lu_test_240 = (X_lu_test_240 - train_mean) / train_std
X_lu_adapt_320 = (X_lu_adapt_320 - r5_train_mean) / r5_train_std
X_lu_test_320 = (X_lu_test_320 - r5_train_mean) / r5_train_std


def seed_metric_row(seed, name, truth, prediction, ids, provenance):
    row = classification_row(name, truth, prediction, ids, provenance['post_samples'])
    row.update({'Seed': seed, 'Best epoch': provenance.get('best_epoch'), 'Best validation loss': provenance.get('best_validation_loss'), 'R4 threshold': provenance.get('r4_threshold', np.nan), 'R6 adaptation epochs': provenance.get('r6_adaptation_epochs', np.nan)})
    return row


seed_rows, seed_provenance = [], []
for seed in TRAINING_SEEDS:
    set_global_seed(seed)
    seed_dir = ARTIFACT_DIR / f'seed_{seed}'
    seed_dir.mkdir(exist_ok=True)
    run_info = {'seed': seed, 'qtdb_train_records': qtdb_train_records, 'qtdb_validation_records': qtdb_val_records, 'ludb_adaptation_records': r6_adapt_records, 'ludb_test_records': r6_test_records, 'qtdb_train_windows_240': len(X_qt_train), 'qtdb_validation_windows_240': len(X_qt_val), 'qtdb_train_windows_320': len(X_qt_train_320), 'qtdb_validation_windows_320': len(X_qt_val_320), 'ludb_adaptation_windows_240': len(X_lu_adapt_240), 'ludb_test_windows_240': len(X_lu_test_240), 'ludb_adaptation_windows_320': len(X_lu_adapt_320), 'ludb_test_windows_320': len(X_lu_test_320), 'qtdb_train_mean_240': train_mean, 'qtdb_train_std_240': train_std, 'qtdb_train_mean_320': r5_train_mean, 'qtdb_train_std_320': r5_train_std, 'r6_adaptation_epochs': R6_ADAPT_EPOCHS}

    r1, r1_info = train_fixed_model(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, nn.CrossEntropyLoss(), seed, 'R1')
    r1_info['post_samples'] = POST
    r1_prediction = predict(r1, X_lu_test_240[:, None])
    seed_rows.append(seed_metric_row(seed, 'R1', Y_lu_test_240, r1_prediction, lu_test_ids_240, r1_info))

    class_counts = np.bincount(Y_qt_train.ravel(), minlength=3).astype(np.float32)
    class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
    class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float32, device=DEVICE)
    r2, r2_info = train_fixed_model(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, FocalLoss(alpha=class_weights), seed, 'R2')
    r2_info['post_samples'] = POST
    seed_rows.append(seed_metric_row(seed, 'R2', Y_lu_test_240, predict(r2, X_lu_test_240[:, None]), lu_test_ids_240, r2_info))

    r3, r3_info = train_fixed_model(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, FocalLoss(), seed, 'R3')
    r3_info['post_samples'] = POST
    r3_prediction = predict(r3, X_lu_test_240[:, None])
    seed_rows.append(seed_metric_row(seed, 'R3', Y_lu_test_240, r3_prediction, lu_test_ids_240, r3_info))

    r4_probabilities = predict_probabilities(r1, X_qt_val[:, None])
    threshold_grid = []
    for threshold in np.arange(0.10, 0.76, 0.05):
        decoded = r4_decode(r4_probabilities, threshold)
        threshold_grid.append((threshold, f1_score(Y_qt_val.ravel(), decoded.ravel(), labels=[0, 1, 2], average='macro', zero_division=0)))
    r4_threshold = float(max(threshold_grid, key=lambda item: item[1])[0])
    r4_info = dict(r1_info, r4_threshold=r4_threshold, post_samples=POST)
    r4_prediction = r4_decode(predict_probabilities(r1, X_lu_test_240[:, None]), r4_threshold)
    seed_rows.append(seed_metric_row(seed, 'R4', Y_lu_test_240, r4_prediction, lu_test_ids_240, r4_info))

    r5, r5_info = train_fixed_model(RPeakGuidedML2().to(DEVICE), X_qt_train_320[:, None], Y_qt_train_320, X_qt_val_320[:, None], Y_qt_val_320, nn.CrossEntropyLoss(), seed, 'R5')
    r5_info['post_samples'] = R5_POST
    seed_rows.append(seed_metric_row(seed, 'R5', Y_lu_test_320, predict(r5, X_lu_test_320[:, None]), lu_test_ids_320, r5_info))

    r6, r6_info = train_fixed_model(RPeakTimeML2().to(DEVICE), time_channel(X_qt_train_320, R5_POST), Y_qt_train_320, time_channel(X_qt_val_320, R5_POST), Y_qt_val_320, nn.CrossEntropyLoss(), seed, 'R6 QTDB')
    r6_adapt_loader = make_loader(time_channel(X_lu_adapt_320, R5_POST), Y_lu_adapt_320, seed, shuffle=True)
    adaptation_optimizer = torch.optim.Adam(r6.parameters(), lr=1e-4)
    for _ in range(R6_ADAPT_EPOCHS):
        run_training_epoch(r6, r6_adapt_loader, nn.CrossEntropyLoss(), adaptation_optimizer)
    r6_info.update({'post_samples': R5_POST, 'r6_adaptation_epochs': R6_ADAPT_EPOCHS})
    seed_rows.append(seed_metric_row(seed, 'R6 adapted', Y_lu_test_320, predict(r6, time_channel(X_lu_test_320, R5_POST)), lu_test_ids_320, r6_info))
    seed_provenance.append(run_info | {'r1': r1_info, 'r2': r2_info, 'r3': r3_info, 'r4': r4_info, 'r5': r5_info, 'r6': r6_info})

seed_metrics = pd.DataFrame(seed_rows)
seed_metrics.to_csv(ARTIFACT_DIR / 'five_seed_metrics.csv', index=False)
(ARTIFACT_DIR / 'five_seed_provenance.json').write_text(json.dumps(seed_provenance, indent=2, default=lambda value: value.item() if isinstance(value, np.generic) else value))
print(seed_metrics[['Seed', 'Experiment', 'P F1', 'T F1', 'Macro F1']].to_string(index=False))

KeyboardInterrupt: 

In [ ]:
def mean_sd_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    critical = float(student_t.ppf((1 + confidence) / 2, len(values) - 1))
    margin = critical * sd / np.sqrt(len(values))
    return {'mean': mean, 'sd': sd, 'ci_low': mean - margin, 'ci_high': mean + margin}

seed_summary_rows = []
for experiment, group in seed_metrics.groupby('Experiment'):
    for metric in ['P F1', 'T F1', 'Macro F1', 'Weighted F1', 'Accuracy']:
        summary = mean_sd_ci(group[metric].to_numpy())
        seed_summary_rows.append({'Experiment': experiment, 'Metric': metric, 'Seeds': len(group), **summary})
seed_summary = pd.DataFrame(seed_summary_rows)
seed_summary.to_csv(ARTIFACT_DIR / 'five_seed_mean_sd_t_ci.csv', index=False)

macro_by_seed = seed_metrics.pivot(index='Seed', columns='Experiment', values='Macro F1')
r3_r6_difference = (macro_by_seed['R6 adapted'] - macro_by_seed['R3']).to_numpy()
r3_r6_summary = pd.DataFrame([{'Comparison': 'R6 adapted - R3', **mean_sd_ci(r3_r6_difference)}])
r3_r6_per_seed = pd.DataFrame({'Seed': macro_by_seed.index, 'R3 Macro F1': macro_by_seed['R3'], 'R6 Macro F1': macro_by_seed['R6 adapted'], 'R6 minus R3': r3_r6_difference})
r3_r6_per_seed.to_csv(ARTIFACT_DIR / 'r3_r6_macro_f1_by_seed.csv', index=False)
r3_r6_summary.to_csv(ARTIFACT_DIR / 'r3_r6_macro_f1_difference_ci.csv', index=False)

print('Training-seed uncertainty: mean, sample SD, and t-based 95% CI across five independent runs.')
display(seed_summary.round(4))
display(r3_r6_per_seed.round(4))
display(r3_r6_summary.round(4))

In [ ]:
# Existing-checkpoint evaluation on the common untouched LUDB test records.
# The arrays were normalized above using QTDB-training statistics.
X240_test = X_lu_test_240
X320_test = X_lu_test_320
results, predictions_by_experiment = [], {}

def score(name, model, features, truth, ids, post, decoder=None):
    probabilities = predict_probabilities(model, features)
    prediction = decoder(probabilities) if decoder else probabilities.argmax(2)
    predictions_by_experiment[name] = (truth, prediction, ids, post)
    results.append(classification_row(name, truth, prediction, ids, post))

for name in ['R1', 'R2', 'R3']:
    model = load_checkpoint(CHECKPOINTS[name])
    score(name, model, X240_test[:, None], Y_lu_test_240, lu_test_ids_240, POST)
score('R4', r4_base, X240_test[:, None], Y_lu_test_240, lu_test_ids_240, POST, lambda p: r4_decode(p, R4_T_THRESHOLD))
r5_model = load_checkpoint(CHECKPOINTS['R5'])
score('R5', r5_model, X320_test[:, None], Y_lu_test_320, lu_test_ids_320, R5_POST)

# The repository has no provenance manifest for the legacy adapted checkpoint.
# Do not present it as a matched result unless its exact adaptation records are verified.
r6_provenance_path = ARTIFACT_DIR / 'r6_adapted_checkpoint_provenance.json'
r6_checkpoint_is_verified = False
if r6_provenance_path.exists():
    r6_provenance = json.loads(r6_provenance_path.read_text())
    r6_checkpoint_is_verified = sorted(r6_provenance.get('ludb_adaptation_records', [])) == sorted(r6_adapt_records)
if r6_checkpoint_is_verified:
    r6_model = load_checkpoint(CHECKPOINTS['R6 adapted'], time=True)
    score('R6 adapted legacy checkpoint', r6_model, time_channel(X320_test, R5_POST), Y_lu_test_320, lu_test_ids_320, R5_POST)
else:
    print('Legacy R6 adapted checkpoint excluded: exact 20-record adaptation provenance is unavailable or does not match the current split.')
matched_metrics = pd.DataFrame(results)
display(matched_metrics.round(4))
matched_metrics.to_csv(ARTIFACT_DIR / 'matched_ludb_test_metrics.csv', index=False)

,Experiment,Records,Windows,Post samples,Accuracy,P precision,P recall,P F1,T precision,T recall,T F1,Macro F1,Weighted F1
0,R1,180,1710,240,0.8780,0.7785,0.8085,0.7932,0.7346,0.7415,0.7381,0.8166,0.8784
1,R2,180,1710,240,0.8161,0.6071,0.9195,0.7313,0.5867,0.7655,0.6643,0.7551,0.8246
2,R3,180,1710,240,0.8637,0.7783,0.7379,0.7576,0.7071,0.7051,0.7061,0.7911,0.8633
3,R4,180,1710,240,0.8377,0.7735,0.4323,0.5547,0.8114,0.5364,0.6459,0.6995,0.8230
4,R5,180,1655,320,0.8641,0.7740,0.7660,0.7700,0.7323,0.7420,0.7371,0.8050,0.8642
5,R6 adapted,180,1655,320,0.8726,0.7565,0.8291,0.7911,0.7427,0.7916,0.7663,0.8230,0.8740


## Fixed-model record bootstrap uncertainty

This section reports record-level bootstrap uncertainty for the existing fixed checkpoints. It is separate from the five independent training-seed uncertainty reported above. Event-boundary analysis is intentionally excluded because the prior implementation used beat-local coordinates without valid global offsets or an effective matching tolerance.

In [ ]:
def record_metric_table(truth, prediction, ids):
    rows = []
    for record in np.unique(ids):
        mask = ids == record
        rows.append({'record': record, 'Macro F1': f1_score(truth[mask].ravel(), prediction[mask].ravel(), labels=[0, 1, 2], average='macro', zero_division=0), 'P F1': f1_score(truth[mask].ravel(), prediction[mask].ravel(), labels=[1], average='macro', zero_division=0), 'T F1': f1_score(truth[mask].ravel(), prediction[mask].ravel(), labels=[2], average='macro', zero_division=0)})
    return pd.DataFrame(rows)


def bootstrap_ci(values, repetitions=2000, seed=2026):
    values = np.asarray(values, dtype=float); values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(repetitions, len(values)), replace=True).mean(axis=1)
    return float(np.mean(values)), float(np.percentile(samples, 2.5)), float(np.percentile(samples, 97.5))

ci_rows = []
for name, (truth, prediction, ids, post) in predictions_by_experiment.items():
    per_record = record_metric_table(truth, prediction, ids)
    for metric in ['Macro F1', 'P F1', 'T F1']:
        mean, low, high = bootstrap_ci(per_record[metric])
        ci_rows.append({'Experiment': name, 'Metric': metric, 'Record mean': mean, '95% CI low': low, '95% CI high': high, 'Records': len(per_record)})
confidence_intervals = pd.DataFrame(ci_rows)
display(confidence_intervals.round(4))
confidence_intervals.to_csv(ARTIFACT_DIR / 'record_bootstrap_confidence_intervals.csv', index=False)
print('These intervals represent record-level bootstrap uncertainty for fixed trained checkpoints, not training-seed uncertainty.')

,Experiment,Metric,Record mean,95% CI low,95% CI high,Records
0,R1,Macro F1,0.7850,0.7629,0.8055,180
1,R1,P F1,0.7005,0.6565,0.7400,180
2,R1,T F1,0.7340,0.7054,0.7614,180
3,R2,Macro F1,0.7470,0.7253,0.7682,180
4,R2,P F1,0.6974,0.6572,0.7342,180
5,R2,T F1,0.6741,0.6485,0.7002,180
6,R3,Macro F1,0.7563,0.7323,0.7786,180
7,R3,P F1,0.6550,0.6092,0.6963,180
8,R3,T F1,0.7014,0.6691,0.7314,180
9,R4,Macro F1,0.6844,0.6640,0.7039,180


,Experiment,Wave,Sensitivity,PPV,Missed,False positives,Onset bias ms,Onset SD ms,Offset bias ms,Offset SD ms
0,R1,P,0.0074,0.0053,13.6389,18.5389,50.1429,19.7990,-33.0000,38.6552
1,R1,T,0.0185,0.0112,12.8500,18.9500,64.8333,57.1045,-58.0741,33.2205
2,R2,P,0.0180,0.0098,13.4778,24.8611,70.6458,18.4193,-32.9375,18.1012
3,R2,T,0.0253,0.0155,12.6889,22.1167,33.0000,32.3290,-64.0071,26.5249
4,R3,P,0.0118,0.0077,13.5889,17.3222,58.8704,46.2852,-42.8889,49.8025
5,R3,T,0.0240,0.0158,12.7389,16.6389,56.1250,37.9627,-46.0350,20.5988
6,R4,P,0.0019,0.0030,13.7056,9.3833,52.8000,NaN,-57.6000,NaN
7,R4,T,0.0000,0.0000,13.1333,8.9556,NaN,NaN,NaN,NaN
8,R5,P,0.0086,0.0066,14.6611,19.0889,75.4917,17.5621,-27.5083,19.9595
9,R5,T,0.0182,0.0109,17.2889,24.2889,64.7444,50.1287,-65.3365,59.1418


QRS event metrics are not reported: the current 3-class target mapping intentionally collapses QRS into background.


## Leakage and decision audit

The checkpoint re-evaluation above does not fit on LUDB. R6's adapted checkpoint is accepted only as an existing artifact; if it is regenerated, adaptation must use `r6_adapt_records` only, for the fixed `R6_ADAPT_EPOCHS`, and the untouched test arrays must not be created until adaptation is complete.

The current target mapping collapses QRS to background, so a separate QRS boundary result requires a new target definition and retraining; this notebook does not relabel the existing models to manufacture that metric.